In [1]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

movies = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')

print('=== MOVIES ===')
print('Shape:', movies.shape)
print('Columns:', movies.columns.tolist())

print("\n=== CREDITS ===")
print("Shape:", credits.shape)
print("Columns:", credits.columns.tolist())

=== MOVIES ===
Shape: (4803, 20)
Columns: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

=== CREDITS ===
Shape: (4803, 4)
Columns: ['movie_id', 'title', 'cast', 'crew']


In [2]:
# Merge the two datasets on movie title
movies = movies.merge(credits, on='title')

print('After merge shape:', movies.shape)
print("\nKey columns we'll use:")
useful = ['title', 'genres', 'keywords', 'cast', 'crew', 'overview', 'vote_average', 'vote_count', 'popularity']
print(movies[useful].head(2))

After merge shape: (4809, 23)

Key columns we'll use:
                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   

                                            keywords  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...   

                                                cast  \
0  [{"cast_id": 242, "character": "Jake Sully", "...   
1  [{"cast_id": 4, "character": "Captain Jack Spa...   

                                                crew  \
0  [{"credit_id": "52fe48009251416c750aca23", "de...   
1  [{"credit_id": "52fe4232c3a36847f800b579", "de...   

                                            overview  vote_average  \
0  In the 22nd century, a paraplegic Marine is di...      

In [3]:
print("=== DATASET OVERVIEW ===")
print(f"Total movies: {len(movies)}")

# Fix: drop NaN before getting date range
dates = movies['release_date'].dropna()
print(f"Date range: {dates.min()} → {dates.max()}")

print(f"\nTop 5 highest rated movies (min 100 votes):")
top = movies[movies['vote_count'] > 100].nlargest(5, 'vote_average')[
    ['title', 'vote_average', 'vote_count']]
print(top.to_string(index=False))

print(f"\nMissing values in key columns:")
useful = ['title', 'genres', 'keywords', 'cast', 
          'crew', 'overview', 'vote_average', 'vote_count']
print(movies[useful].isnull().sum())

print(f"\nSample genres column (raw):")
print(movies['genres'][0])
print("\nNotice: genres is stored as a JSON string — we need to parse it!")

=== DATASET OVERVIEW ===
Total movies: 4809
Date range: 1916-09-04 → 2017-02-03

Top 5 highest rated movies (min 100 votes):
                   title  vote_average  vote_count
The Shawshank Redemption           8.5        8205
           The Godfather           8.4        5893
              Fight Club           8.3        9413
        Schindler's List           8.3        4329
           Spirited Away           8.3        3840

Missing values in key columns:
title           0
genres          0
keywords        0
cast            0
crew            0
overview        3
vote_average    0
vote_count      0
dtype: int64

Sample genres column (raw):
[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]

Notice: genres is stored as a JSON string — we need to parse it!


In [4]:
import ast

def parse_names(obj, key='name', limit=None):
    """Extract names from JSON string column"""
    try:
        items = ast.literal_eval(obj)
        names = [i[key] for i in items]
        return names[:limit] if limit else names
    except:
        return []

def get_director(crew_obj):
    """Extract director name from crew JSON"""
    try:
        crew = ast.literal_eval(crew_obj)
        for member in crew:
            if member['job'] == 'Director':
                return member['name']
        return ''
    except:
        return ''

# Apply parsers
movies['genres_list']   = movies['genres'].apply(parse_names)
movies['keywords_list'] = movies['keywords'].apply(parse_names)
movies['cast_list']     = movies['cast'].apply(
    lambda x: parse_names(x, limit=3))  # top 3 cast only
movies['director']      = movies['crew'].apply(get_director)

# Verify
print("Sample parsed data for Avatar:")
print(f"  Genres:   {movies['genres_list'][0]}")
print(f"  Keywords: {movies['keywords_list'][0][:5]}")
print(f"  Cast:     {movies['cast_list'][0]}")
print(f"  Director: {movies['director'][0]}")

Sample parsed data for Avatar:
  Genres:   ['Action', 'Adventure', 'Fantasy', 'Science Fiction']
  Keywords: ['culture clash', 'future', 'space war', 'space colony', 'society']
  Cast:     ['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']
  Director: James Cameron


In [5]:
def clean_name(name):
    """Remove spaces so 'Sam Worthington' → 'SamWorthington'
    This prevents 'Sam' from matching other Sams"""
    return name.replace(' ', '')

def build_tags(row):
    genres   = [clean_name(g) for g in row['genres_list']]
    keywords = [clean_name(k) for k in row['keywords_list'][:5]]
    cast     = [clean_name(c) for c in row['cast_list']]
    director = [clean_name(row['director'])] * 3  # weight director 3x
    overview = row['overview'].split() if pd.notna(row['overview']) else []
    
    return ' '.join(genres + keywords + cast + director + overview)

movies['tags'] = movies.apply(build_tags, axis=1)
movies['tags'] = movies['tags'].str.lower()

print("Sample tags for Avatar:")
print(movies['tags'][0][:200])
print("\nSample tags for The Dark Knight:")
dk = movies[movies['title'] == 'The Dark Knight'].iloc[0]
print(movies.loc[movies['title'] == 'The Dark Knight', 'tags'].values[0][:200])

Sample tags for Avatar:
action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society samworthington zoesaldana sigourneyweaver jamescameron jamescameron jamescameron in the 22nd century, a paraple

Sample tags for The Dark Knight:
drama action crime thriller dccomics crimefighter secretidentity scarecrow sadism christianbale heathledger aaroneckhart christophernolan christophernolan christophernolan batman raises the stakes in 


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# WHY TF-IDF here instead of CountVectorizer:
# Common words like "love", "man" appear in many movies
# TF-IDF downweights them and upweights distinctive words
# This gives better similarity scores

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['tags'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Each movie → vector of {tfidf_matrix.shape[1]} features")

# Compute cosine similarity between ALL movies
# This creates a 4809 x 4809 matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"\nCosine similarity matrix shape: {cosine_sim.shape}")
print(f"Memory: {cosine_sim.nbytes / 1024**2:.1f} MB")

# Build index for fast lookup
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()
print(f"\nMovies indexed: {len(indices)}")

TF-IDF matrix shape: (4809, 5000)
Each movie → vector of 5000 features

Cosine similarity matrix shape: (4809, 4809)
Memory: 176.4 MB

Movies indexed: 4809


In [7]:
def recommend_content(title, n=10):
    """Recommend movies based on content similarity"""
    if title not in indices:
        return f"Movie '{title}' not found in dataset"
    
    idx = indices[title]
    
    # Get similarity scores for this movie vs all others
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip the first one (it's the movie itself, similarity=1.0)
    sim_scores = sim_scores[1:n+1]
    
    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]
    similarity    = [round(i[1], 3) for i in sim_scores]
    
    result = movies.iloc[movie_indices][
        ['title', 'genres_list', 'vote_average', 'director']].copy()
    result['similarity'] = similarity
    result['genres_list'] = result['genres_list'].apply(
        lambda x: ', '.join(x[:3]))
    
    return result.reset_index(drop=True)

# Test it!
print("=== Recommendations for: The Dark Knight ===")
print(recommend_content('The Dark Knight').to_string())

print("\n=== Recommendations for: The Avengers ===")
print(recommend_content('The Avengers').to_string())

print("\n=== Recommendations for: Inception ===")
print(recommend_content('Inception').to_string())

=== Recommendations for: The Dark Knight ===
                                     title                        genres_list  vote_average           director  similarity
0                    The Dark Knight Rises               Action, Crime, Drama           7.6  Christopher Nolan       0.516
1                            Batman Begins               Action, Crime, Drama           7.5  Christopher Nolan       0.502
2                             The Prestige           Drama, Mystery, Thriller           8.0  Christopher Nolan       0.328
3                                 Insomnia           Crime, Mystery, Thriller           6.8  Christopher Nolan       0.300
4                             Interstellar  Adventure, Drama, Science Fiction           8.1  Christopher Nolan       0.284
5                                Inception  Action, Thriller, Science Fiction           8.1  Christopher Nolan       0.279
6                                  Memento                  Mystery, Thriller           8.1  C

In [8]:
# Check: do recommendations make sense genre-wise?
def genre_overlap(title, n=10):
    """What % of recommended movies share at least one genre?"""
    source_genres = set(
        movies[movies['title']==title]['genres_list'].values[0])
    recs = recommend_content(title, n)
    
    overlaps = []
    for _, row in recs.iterrows():
        rec_genres = set(row['genres_list'].split(', '))
        overlap = len(source_genres & rec_genres) > 0
        overlaps.append(overlap)
    
    return f"{title}: {sum(overlaps)}/{n} recommendations share a genre"

print("=== Genre Overlap Check ===")
for movie in ['The Dark Knight', 'Inception', 'The Avengers', 
              'Toy Story', 'The Godfather']:
    try:
        print(genre_overlap(movie))
    except:
        print(f"{movie}: not found")

=== Genre Overlap Check ===
The Dark Knight: 10/10 recommendations share a genre
Inception: 8/10 recommendations share a genre
The Avengers: 10/10 recommendations share a genre
Toy Story: 8/10 recommendations share a genre
The Godfather: 9/10 recommendations share a genre


In [14]:
# For collaborative filtering we need user ratings
# TMDB doesn't have individual user ratings — only aggregate vote_average
# We'll simulate realistic user ratings using MovieLens-style approach
# This is a common technique when individual ratings aren't available

np.random.seed(42)

# Create a realistic user-movie ratings dataset
n_users = 500
n_movies = 200

# Use top 200 most popular movies
top_movies = movies.nlargest(200, 'vote_count')[
    ['title', 'vote_average']].reset_index(drop=True)

# Simulate ratings: each user rates ~40 movies
# Ratings are influenced by the movie's actual vote_average (realistic!)
ratings_data = []

for user_id in range(1, n_users + 1):
    # Each user rates a random subset of movies
    n_rated = np.random.randint(20, 60)
    rated_movies = np.random.choice(n_movies, n_rated, replace=False)
    
    for movie_idx in rated_movies:
        true_rating = top_movies.iloc[movie_idx]['vote_average']
        # Add personal preference noise
        user_rating = true_rating + np.random.normal(0, 1.2)
        user_rating = np.clip(round(user_rating * 2) / 2, 1, 10)
        
        ratings_data.append({
            'userId':  user_id,
            'title':   top_movies.iloc[movie_idx]['title'],
            'rating':  user_rating,
            'movieId': movie_idx
        })

ratings_df = pd.DataFrame(ratings_data)

print(f"Ratings dataset: {len(ratings_df):,} ratings")
print(f"Users: {ratings_df['userId'].nunique()}")
print(f"Movies: {ratings_df['title'].nunique()}")
print(f"Avg ratings per user: {len(ratings_df)/n_users:.0f}")
print(f"\nRating distribution:")
print(ratings_df['rating'].value_counts().sort_index())

Ratings dataset: 19,884 ratings
Users: 500
Movies: 200
Avg ratings per user: 40

Rating distribution:
rating
2.0        3
2.5       17
3.0       20
3.5       64
4.0      187
4.5      418
5.0      742
5.5     1217
6.0     1881
6.5     2540
7.0     2845
7.5     2831
8.0     2600
8.5     1911
9.0     1281
9.5      699
10.0     628
Name: count, dtype: int64


In [15]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate, train_test_split
from surprise import accuracy

# WHY SVD (Singular Value Decomposition):
# The user-movie matrix is huge and mostly empty (sparse)
# Most users haven't rated most movies
# SVD finds hidden "latent factors" — like "action lover", "art film fan"
# It fills in the missing ratings by learning these hidden preferences

reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(
    ratings_df[['userId', 'movieId', 'rating']], reader)

# Cross-validate to check model quality
print("=== Cross-Validation (5-fold) ===")
results = cross_validate(SVD(n_factors=50, random_state=42),
                          data, measures=['RMSE', 'MAE'],
                          cv=5, verbose=True)

print(f"\nMean RMSE: {results['test_rmse'].mean():.4f}")
print(f"Mean MAE:  {results['test_mae'].mean():.4f}")

=== Cross-Validation (5-fold) ===
Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    1.2205  1.2465  1.2199  1.2075  1.2280  1.2245  0.0128  
MAE (testset)     0.9876  0.9990  0.9770  0.9744  0.9807  0.9838  0.0088  
Fit time          0.04    0.03    0.03    0.03    0.03    0.03    0.00    
Test time         0.06    0.00    0.01    0.01    0.00    0.02    0.02    

Mean RMSE: 1.2245
Mean MAE:  0.9838


In [16]:
# Train on full dataset
trainset = data.build_full_trainset()
svd_model = SVD(n_factors=50, random_state=42)
svd_model.fit(trainset)

def recommend_collaborative(user_id, n=10):
    """Recommend movies a user hasn't seen yet"""
    # Movies this user has already rated
    rated = ratings_df[ratings_df['userId']==user_id]['movieId'].tolist()
    
    # Predict ratings for all unrated movies
    unrated = [mid for mid in range(n_movies) if mid not in rated]
    
    predictions = []
    for movie_id in unrated:
        pred = svd_model.predict(user_id, movie_id)
        title = top_movies.iloc[movie_id]['title']
        predictions.append((title, round(pred.est, 2)))
    
    # Sort by predicted rating
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    result = pd.DataFrame(predictions[:n],
                           columns=['title', 'predicted_rating'])
    # Add movie info
    result = result.merge(
        movies[['title', 'genres_list', 'vote_average', 'director']],
        on='title', how='left')
    result['genres_list'] = result['genres_list'].apply(
        lambda x: ', '.join(x[:3]) if isinstance(x, list) else '')
    return result

# Test for different users
for uid in [1, 42, 100]:
    print(f"\n=== Top 5 Recommendations for User {uid} ===")
    recs = recommend_collaborative(uid, n=5)
    print(recs[['title', 'predicted_rating',
                'genres_list']].to_string(index=False))


=== Top 5 Recommendations for User 1 ===
           title  predicted_rating                        genres_list
Schindler's List              8.64                Drama, History, War
12 Years a Slave              8.60                     Drama, History
     The Shining              8.59                   Horror, Thriller
       Star Wars              8.51 Adventure, Action, Science Fiction
    Forrest Gump              8.45             Comedy, Drama, Romance

=== Top 5 Recommendations for User 42 ===
                   title  predicted_rating                       genres_list
                Whiplash              8.08                             Drama
The Shawshank Redemption              7.95                      Drama, Crime
         The Dark Knight              7.93              Drama, Action, Crime
               Inception              7.89 Action, Thriller, Science Fiction
            The Prestige              7.83          Drama, Mystery, Thriller

=== Top 5 Recommendations for Us

In [17]:
def recommend_hybrid(title, user_id, n=10,
                     content_weight=0.6, collab_weight=0.4):
    """
    Combine content-based and collaborative filtering
    content_weight + collab_weight = 1.0
    """
    # --- Content-based scores ---
    if title not in indices:
        return f"Movie '{title}' not found"

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_dict = {i: score for i, score in sim_scores}

    # --- Collaborative scores ---
    collab_preds = {}
    for movie_id in range(n_movies):
        t = top_movies.iloc[movie_id]['title']
        pred = svd_model.predict(user_id, movie_id).est
        collab_preds[t] = pred

    # --- Combine ---
    hybrid_scores = []
    for i, row in movies.iterrows():
        t = row['title']
        content_score = sim_dict.get(i, 0)
        # Normalize collab score to 0-1
        collab_score = (collab_preds.get(t, 5) - 1) / 9
        hybrid = (content_weight * content_score +
                  collab_weight * collab_score)
        hybrid_scores.append((i, t, hybrid))

    hybrid_scores.sort(key=lambda x: x[2], reverse=True)

    # Remove the input movie itself
    hybrid_scores = [(i, t, s) for i, t, s in hybrid_scores
                     if t != title][:n]

    result = pd.DataFrame(hybrid_scores,
                           columns=['idx', 'title', 'hybrid_score'])
    result = result.merge(
        movies[['title', 'genres_list', 'vote_average']],
        on='title', how='left')
    result['genres_list'] = result['genres_list'].apply(
        lambda x: ', '.join(x[:3]) if isinstance(x, list) else '')
    result['hybrid_score'] = result['hybrid_score'].round(3)
    return result[['title', 'hybrid_score',
                   'genres_list', 'vote_average']]

print("=== Hybrid Recommendations ===")
print("Movie: Inception | User: 42\n")
print(recommend_hybrid('Inception', user_id=42).to_string(index=False))

=== Hybrid Recommendations ===
Movie: Inception | User: 42

                  title  hybrid_score                        genres_list  vote_average
           The Prestige         0.532           Drama, Mystery, Thriller           8.0
           Interstellar         0.506  Adventure, Drama, Science Fiction           8.1
        The Dark Knight         0.476               Drama, Action, Crime           8.2
                Memento         0.469                  Mystery, Thriller           8.1
          Batman Begins         0.467               Action, Crime, Drama           7.5
  The Dark Knight Rises         0.427               Action, Crime, Drama           7.6
               Insomnia         0.395           Crime, Mystery, Thriller           6.8
Guardians of the Galaxy         0.324 Action, Science Fiction, Adventure           7.9
       12 Years a Slave         0.321                     Drama, History           7.9
               Whiplash         0.315                              Dra

In [18]:
import pickle
import json

# Save cosine similarity matrix
with open('cosine_sim.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)

# Save SVD model
with open('svd_model.pkl', 'wb') as f:
    pickle.dump(svd_model, f)

# Save movie data (clean version for app)
movies_clean = movies[['title', 'genres_list', 'vote_average', 
                        'vote_count', 'director', 'overview',
                        'tags']].copy()
movies_clean['genres_str'] = movies_clean['genres_list'].apply(
    lambda x: ', '.join(x[:3]) if isinstance(x, list) else '')
movies_clean.to_csv('movies_clean.csv', index=False)

# Save indices mapping
indices_dict = indices.to_dict()
with open('movie_indices.json', 'w') as f:
    json.dump(indices_dict, f)

# Save top_movies and ratings for collaborative filtering
top_movies.to_csv('top_movies.csv', index=False)
ratings_df.to_csv('ratings_df.csv', index=False)

print("✅ All files saved!")
print("Files:", ['cosine_sim.pkl', 'svd_model.pkl', 'movies_clean.csv',
                 'movie_indices.json', 'top_movies.csv', 'ratings_df.csv'])

✅ All files saved!
Files: ['cosine_sim.pkl', 'svd_model.pkl', 'movies_clean.csv', 'movie_indices.json', 'top_movies.csv', 'ratings_df.csv']


In [19]:
from sklearn.decomposition import TruncatedSVD
import scipy.sparse as sp

user_movie_matrix = ratings_df.pivot_table(
    index='userId', columns='movieId',
    values='rating', fill_value=0)

svd_sklearn = TruncatedSVD(n_components=50, random_state=42)
matrix_sparse = sp.csr_matrix(user_movie_matrix.values)
user_factors = svd_sklearn.fit_transform(matrix_sparse)
item_factors = svd_sklearn.components_

predicted_ratings = np.dot(user_factors, item_factors)
predicted_df = pd.DataFrame(
    predicted_ratings,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.columns)

predicted_df.to_csv('predicted_ratings.csv')
print("✅ Saved!")

✅ Saved!
